# MTG Deck Card Predictor — Optuna Hyperparameter Sweep

Automated search for optimal hyperparameters using Bayesian optimization.
Optuna tries many configurations, prunes unpromising trials early, and reports
which parameters matter most.

**Architecture**: HeteroGNN + Cross-Attention Autoregressive Deck Constructor with Gumbel-Softmax.
Card selection CE + Count prediction CE loss. Archetype-holdout CV (train on 13, validate on 3).

**Primary metric**: Jaccard similarity between predicted and actual decklist.

**Runtime**: Select `Runtime > Change runtime type > T4 GPU` before running.

**Expected time**: ~2–3 hours for 50 trials on T4.

In [ ]:
# Mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/mtg-graph'
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

!pip install -q torch-geometric sentence-transformers python-dotenv \
    beautifulsoup4 lxml spacy tqdm pyarrow optuna

print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')

In [ ]:
# ── Search Space Configuration (edit these) ──

N_TRIALS = 200
STUDY_NAME = 'deck-sweep'

SEARCH_SPACE = {
    'd_model':           [64, 128, 256],
    'd_message':         [64, 128, 256],
    'd_count':           [8, 16, 32],
    'num_attn_heads':    [2, 4, 8],
    'num_gnn_layers':    {'low': 1, 'high': 4},
    'dropout':           {'low': 0.05, 'high': 0.30, 'step': 0.05},
    'learning_rate':     {'low': 1e-5, 'high': 1e-3},
    'weight_decay':      {'low': 1e-6, 'high': 1e-3},
    'count_loss_weight': {'low': 0.1, 'high': 3.0, 'step': 0.1},
    'gumbel_tau_start':  [0.5, 1.0, 2.0],
    'gumbel_tau_min':    [0.05, 0.1, 0.2],
    'gumbel_decay':      {'low': 0.005, 'high': 0.05, 'step': 0.005},
    'warmup_epochs':     [0, 3, 5, 10],
    'lr_scheduler':      ['cosine', 'linear', 'none'],
}

FIXED_PARAMS = {
    'patience': 10,
    'num_epochs': 100,
    'recency_days': 30,
    'n_val_archetypes': 3,
    'n_cv_folds': 5,
    'batch_size': 0,
    'val_every': 10,
}

N_STARTUP_TRIALS = 10
N_WARMUP_STEPS = 3

print(f'Search: {N_TRIALS} trials over {len(SEARCH_SPACE)} parameters')
print(f'Pruning: starts after {N_STARTUP_TRIALS} trials, warmup {N_WARMUP_STEPS} steps')
print(f'Fixed: {FIXED_PARAMS}')

## Run Sweep

In [ ]:
import optuna
import time
import logging
import shutil
import gc
from pathlib import Path

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger('src.train_deck').setLevel(logging.WARNING)

from src.train_deck import train_deck

# Use Colab's local disk for intermediate trial results (fast, no Drive I/O)
LOCAL_RESULTS = Path('/content/sweep_tmp')
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)

trial_results = {}


def objective(trial):
    """Optuna objective: sample hyperparameters, train, return best val Jaccard."""
    params = dict(FIXED_PARAMS)

    params['d_model'] = trial.suggest_categorical('d_model', SEARCH_SPACE['d_model'])
    params['d_message'] = trial.suggest_categorical('d_message', SEARCH_SPACE['d_message'])
    params['d_count'] = trial.suggest_categorical('d_count', SEARCH_SPACE['d_count'])
    params['num_attn_heads'] = trial.suggest_categorical('num_attn_heads', SEARCH_SPACE['num_attn_heads'])
    params['num_gnn_layers'] = trial.suggest_int('num_gnn_layers', **SEARCH_SPACE['num_gnn_layers'])
    params['dropout'] = trial.suggest_float('dropout', **SEARCH_SPACE['dropout'])
    params['learning_rate'] = trial.suggest_float('learning_rate', **SEARCH_SPACE['learning_rate'], log=True)
    params['weight_decay'] = trial.suggest_float('weight_decay', **SEARCH_SPACE['weight_decay'], log=True)
    params['count_loss_weight'] = trial.suggest_float('count_loss_weight', **SEARCH_SPACE['count_loss_weight'])
    params['gumbel_tau_start'] = trial.suggest_categorical('gumbel_tau_start', SEARCH_SPACE['gumbel_tau_start'])
    params['gumbel_tau_min'] = trial.suggest_categorical('gumbel_tau_min', SEARCH_SPACE['gumbel_tau_min'])
    params['gumbel_decay'] = trial.suggest_float('gumbel_decay', **SEARCH_SPACE['gumbel_decay'])
    params['warmup_epochs'] = trial.suggest_categorical('warmup_epochs', SEARCH_SPACE['warmup_epochs'])
    params['lr_scheduler'] = trial.suggest_categorical('lr_scheduler', SEARCH_SPACE['lr_scheduler'])

    # Validate d_model is divisible by num_attn_heads
    if params['d_model'] % params['num_attn_heads'] != 0:
        raise optuna.TrialPruned()

    # Write intermediate results to local disk, not Google Drive
    params['results_base'] = LOCAL_RESULTS

    start = time.time()
    try:
        result = train_deck(device=device, trial=trial, **params)
    except torch.cuda.OutOfMemoryError:
        gc.collect()
        torch.cuda.empty_cache()
        print(f'  Trial {trial.number}: OOM. Skipping.')
        raise optuna.TrialPruned()

    elapsed = time.time() - start

    fold_0 = result['fold_results'][0]
    best_metric = fold_0['best_val_metric']

    val_hist = fold_0.get('val_metrics_history', [])
    last_val = val_hist[-1] if val_hist else {}
    precision = last_val.get('precision', 0)

    trial_results[trial.number] = {
        'result': result,
        'best_val_metric': best_metric,
        'best_epoch': fold_0['best_epoch'],
        'precision': precision,
        'elapsed_s': elapsed,
    }

    print(f'  Trial {trial.number}: Jacc={best_metric:.4f} Prec={precision:.3f} '
          f'(epoch {fold_0["best_epoch"]}, {elapsed:.0f}s) '
          f'| dm={params["d_model"]} dg={params["d_message"]} dc={params["d_count"]} '
          f'ah={params["num_attn_heads"]} '
          f'ly={params["num_gnn_layers"]} '
          f'lr={params["learning_rate"]:.1e} do={params["dropout"]:.2f} '
          f'clw={params["count_loss_weight"]:.1f} '
          f'tau={params["gumbel_tau_start"]}->{params["gumbel_tau_min"]}')

    del result['model']
    gc.collect()
    torch.cuda.empty_cache()

    return best_metric


pruner = optuna.pruners.MedianPruner(
    n_startup_trials=N_STARTUP_TRIALS,
    n_warmup_steps=N_WARMUP_STEPS,
)
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    pruner=pruner,
)

print(f'Starting {N_TRIALS}-trial sweep...')
print(f'Intermediate results: {LOCAL_RESULTS} (local disk)\n')
sweep_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, catch=(Exception,))
sweep_elapsed = time.time() - sweep_start

n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
n_failed = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

print(f'\n{"=" * 60}')
print(f'Sweep complete: {len(study.trials)} trials in {sweep_elapsed/60:.1f} min')
print(f'  Completed: {n_complete}  Pruned: {n_pruned}  Failed: {n_failed}')
print(f'\nBest trial #{study.best_trial.number}:')
print(f'  Jaccard = {study.best_value:.4f}')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
import matplotlib.pyplot as plt

# Optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Trial values over time
ax = axes[0]
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
trial_nums = [t.number for t in completed]
trial_vals = [t.value for t in completed]
best_so_far = [max(trial_vals[:i+1]) for i in range(len(trial_vals))]

ax.scatter(trial_nums, trial_vals, alpha=0.5, s=20, label='Trial value')
ax.plot(trial_nums, best_so_far, 'r-', linewidth=2, label='Best so far')

pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
if pruned:
    ax.scatter([t.number for t in pruned], [0] * len(pruned),
              marker='x', color='gray', alpha=0.3, s=15, label='Pruned')

ax.set_xlabel('Trial'); ax.set_ylabel('Jaccard')
ax.set_title('Optimization History'); ax.legend(); ax.grid(True, alpha=0.3)

# 2. Parameter importance
ax = axes[1]
try:
    importance = optuna.importance.get_param_importances(study)
    names = list(importance.keys())
    values = list(importance.values())
    colors = plt.cm.viridis([v / max(values) for v in values])
    ax.barh(names, values, color=colors)
    ax.set_xlabel('Importance'); ax.set_title('Hyperparameter Importance')
    ax.grid(True, alpha=0.3, axis='x')
except Exception as e:
    ax.text(0.5, 0.5, f'Importance unavailable:\n{e}',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

# Print all completed trials sorted by value
print(f'\nAll completed trials (sorted by Jaccard):')
print(f'{"#":>4}  {"Jacc":>7}  {"Prec":>6}  {"dm":>4}  {"dc":>4}  {"ah":>3}  {"ly":>3}  {"lr":>10}  {"do":>5}  '
      f'{"clw":>5}  {"tau0":>5}  {"tauN":>5}  {"tdec":>6}  {"wu":>3}  {"sched":>7}')
print('-' * 115)
for t in sorted(completed, key=lambda t: t.value, reverse=True):
    p = t.params
    tr = trial_results.get(t.number, {})
    prec = tr.get('precision', 0)
    print(f'{t.number:4d}  {t.value:7.4f}  {prec:6.3f}  {p["d_model"]:4d}  {p["d_count"]:4d}  '
          f'{p["num_attn_heads"]:3d}  {p["num_gnn_layers"]:3d}  '
          f'{p["learning_rate"]:10.1e}  {p["dropout"]:5.2f}  '
          f'{p["count_loss_weight"]:5.1f}  {p["gumbel_tau_start"]:5.2f}  {p["gumbel_tau_min"]:5.2f}  '
          f'{p["gumbel_decay"]:6.3f}  {p["warmup_epochs"]:3d}  {p["lr_scheduler"]:>7s}')

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
from src.train_deck import evaluate_deck
from src.visualize_deck import main as generate_deck_dashboard

# ── Configuration ──
TOP_N = 5  # How many top models to save (set to None to save all completed)

# ── Create sweep output directory on Google Drive (final results only) ──
sweep_ts = datetime.now().strftime('%Y-%m-%d_%H%M%S')
sweep_dir = Path(f'results/sweeps/{sweep_ts}')
sweep_dir.mkdir(parents=True, exist_ok=True)

# ── Identify top trials ──
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
ranked = sorted(completed, key=lambda t: t.value, reverse=True)
top_trials = ranked[:TOP_N] if TOP_N else ranked

print(f'Saving top {len(top_trials)} models to Google Drive: {sweep_dir}/\n')

# ── Save each top model (copy from local disk to Drive) ──
for rank, trial in enumerate(top_trials, 1):
    tr = trial_results.get(trial.number)
    if tr is None:
        print(f'  Skipping trial {trial.number} — results not in memory')
        continue

    result = tr['result']
    fold_0 = result['fold_results'][0]
    p = trial.params

    # Descriptive folder name: rank, trial#, score, key params
    folder_name = (
        f'rank{rank:02d}_trial{trial.number:02d}'
        f'_jacc-{trial.value:.3f}'
        f'_dm{p["d_model"]}_ly{p["num_gnn_layers"]}'
        f'_lr{p["learning_rate"]:.1e}_do{p["dropout"]:.2f}'
    )
    model_dir = sweep_dir / folder_name
    model_dir.mkdir(parents=True, exist_ok=True)

    # Copy model checkpoint from local disk to Drive
    src_model = Path(fold_0['model_path'])
    if src_model.exists():
        shutil.copy2(src_model, model_dir / 'model_fold0.pt')

    # Copy training log from local disk to Drive
    src_log = result['run_dir'] / 'training_log.json'
    if src_log.exists():
        shutil.copy2(src_log, model_dir / 'training_log.json')

    # Run full evaluation
    logging.getLogger('src.train_deck').setLevel(logging.WARNING)
    evaluate_deck(result)

    # Generate dashboard (reads training_log.json for HP, or infers from checkpoint)
    generate_deck_dashboard(run_dir=model_dir, fold_idx=0)

    # Save config JSON with full hyperparameters + metrics
    config = {
        'rank': rank,
        'trial_number': trial.number,
        'best_epoch': fold_0['best_epoch'],
        'training_time_s': tr['elapsed_s'],
        'hyperparameters': result['hp'],
        'val_archetypes': fold_0['val_archetypes'],
        'train_archetypes': fold_0['train_archetypes'],
        'best_val_jaccard': trial.value,
        'training_curves': {
            'train_losses': fold_0['train_losses'],
            'val_metrics_history': fold_0['val_metrics_history'],
        },
    }
    with open(model_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2, default=str)

    print(f'  #{rank} Trial {trial.number}: Jaccard={trial.value:.3f} '
          f'(epoch {fold_0["best_epoch"]}, {tr["elapsed_s"]:.0f}s) '
          f'| val={", ".join(fold_0["val_archetypes"])} -> {folder_name}/')

# ── Save sweep summary JSON ──
sweep_summary = {
    'timestamp': datetime.now(timezone.utc).isoformat(),
    'study_name': STUDY_NAME,
    'n_trials': len(study.trials),
    'n_completed': len(completed),
    'n_pruned': len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]),
    'elapsed_minutes': sweep_elapsed / 60,
    'top_n_saved': len(top_trials),
    'best_trial': {
        'number': study.best_trial.number,
        'value': study.best_value,
        'params': study.best_params,
    },
    'fixed_params': FIXED_PARAMS,
    'search_space': {k: str(v) for k, v in SEARCH_SPACE.items()},
    'all_trials': [
        {
            'number': t.number,
            'state': t.state.name,
            'value': t.value if t.value is not None else None,
            'params': t.params,
            'duration_s': (t.datetime_complete - t.datetime_start).total_seconds()
                if t.datetime_complete and t.datetime_start else None,
        }
        for t in study.trials
    ],
}

# Add parameter importance
try:
    importance = optuna.importance.get_param_importances(study)
    sweep_summary['param_importance'] = {k: float(v) for k, v in importance.items()}
except Exception:
    pass

with open(sweep_dir / 'sweep_summary.json', 'w') as f:
    json.dump(sweep_summary, f, indent=2)

print(f'\n{"=" * 60}')
print(f'Sweep saved to Google Drive: {sweep_dir}/')
print(f'  sweep_summary.json — full sweep results ({len(study.trials)} trials)')
print(f'  {len(top_trials)} model directories — each with model, config, dashboard')

## Cleanup

Delete the local temporary directory used for intermediate trial results.

In [ ]:
# Clean up local temporary trial directories
# All top models have already been copied to Google Drive in the sweep directory.

import shutil
from pathlib import Path

local_dir = Path('/content/sweep_tmp')
if local_dir.exists():
    n_dirs = len(list(local_dir.rglob('*')))
    shutil.rmtree(local_dir, ignore_errors=True)
    print(f'Cleaned up local temp directory: {local_dir} ({n_dirs} items)')
else:
    print('No local temp directory to clean.')

print(f'\nFinal results on Google Drive: {sweep_dir}/')
!ls "{sweep_dir}"